In [2]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import HuggingFaceSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

# LLD-MMRI

completed!

In [3]:
DATASET_NAME = "LLD-MMRI"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



21:06:23 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:78


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver'
]

In [4]:
description = """
LLD-MMRI dataset contains diverse liver lesions from 498 unique patients, including hepatocellular carcinoma, intrahepatic cholangiocarcinoma, liver metastases (HM), hepatic cysts (HC), hepatic hemangioma, focal nodular hyperplasia, and hepatic abscess.

Each lesion has eight MRI scans: non-contrast, arterial, venous, delay, T2-weighted imaging, diffusion-weighted imaging, T1 in-phase, and T1 out-of-phase, resulting in 3984 cases in total.

We annotated all the 3984 lesions with MedSAM2 in a human-in-the-loop pipeline.

"""

In [5]:
source = HuggingFaceSource(
    file_type="nifti",
    repo_id="wanglab/LLD-MMRI-MedSAM2",
    description=description,
    post_download=["unzip"]
)

In [6]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [7]:

print(indexed_datasets.source_config(DATASET_NAME))

HuggingFaceSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='huggingface',
    repo_id='wanglab/LLD-MMRI-MedSAM2',
    post_download=['unzip'],
    description='\nLLD-MMRI dataset contains diverse liver lesions from 498 unique patients, including 
hepatocellular carcinoma, intrahepatic cholangiocarcinoma, liver metastases (HM), hepatic cysts (HC), hepatic 
hemangioma, focal nodular hyperplasia, and hepatic abscess.\n\nEach lesion has eight MRI scans: non-contrast, 
arterial, venous, delay, T2-weighted imaging, diffusion-weighted imaging, T1 in-phase, and T1 out-of-phase, 
resulting in 3984 cases in total.\n\nWe annotated all the 3984 lesions with MedSAM2 in a human-in-the-loop 
pipeline.\n\n'
)

In [8]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
#downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

In [9]:
crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="images/{SampleID}_{_temp}_{Phase}_{_temp2}.nii.gz",
    mask_name_pattern="labels/{SampleID}_{_temp}_{Phase}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()


In [10]:
index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

index_df



,SampleID,_temp,Phase,_temp2,filepath,file_type,reference_id,reference_scan
0,MR-391135,1,C+A,0.0,images/MR-391135_1_C+A_0000.nii.gz,scan,MR-391135_1_C+A,NaN
1,MR-391135,1,C+Delay,0.0,images/MR-391135_1_C+Delay_0000.nii.gz,scan,MR-391135_1_C+Delay,NaN
2,MR-391135,1,C+V,0.0,images/MR-391135_1_C+V_0000.nii.gz,scan,MR-391135_1_C+V,NaN
3,MR-391135,1,C-pre,0.0,images/MR-391135_1_C-pre_0000.nii.gz,scan,MR-391135_1_C-pre,NaN
4,MR-391135,1,DWI,0.0,images/MR-391135_1_DWI_0000.nii.gz,scan,MR-391135_1_DWI,NaN
...,...,...,...,...,...,...,...,...
7963,MR99753,3,C-pre,NaN,labels/MR99753_3_C-pre.nii.gz,mask,MR99753_3_C-pre,images/MR99753_3_C-pre_0000.nii.gz
7964,MR99753,3,DWI,NaN,labels/MR99753_3_DWI.nii.gz,mask,MR99753_3_DWI,images/MR99753_3_DWI_0000.nii.gz
7965,MR99753,3,InPhase,NaN,labels/MR99753_3_InPhase.nii.gz,mask,MR99753_3_InPhase,images/MR99753_3_InPhase_0000.nii.gz
7966,MR99753,3,OutPhase,NaN,labels/MR99753_3_OutPhase.nii.gz,mask,MR99753_3_OutPhase,images/MR99753_3_OutPhase_0000.nii.gz


In [11]:
index_df

,SampleID,_temp,Phase,_temp2,filepath,file_type,reference_id,reference_scan
0,MR-391135,1,C+A,0.0,images/MR-391135_1_C+A_0000.nii.gz,scan,MR-391135_1_C+A,NaN
1,MR-391135,1,C+Delay,0.0,images/MR-391135_1_C+Delay_0000.nii.gz,scan,MR-391135_1_C+Delay,NaN
2,MR-391135,1,C+V,0.0,images/MR-391135_1_C+V_0000.nii.gz,scan,MR-391135_1_C+V,NaN
3,MR-391135,1,C-pre,0.0,images/MR-391135_1_C-pre_0000.nii.gz,scan,MR-391135_1_C-pre,NaN
4,MR-391135,1,DWI,0.0,images/MR-391135_1_DWI_0000.nii.gz,scan,MR-391135_1_DWI,NaN
...,...,...,...,...,...,...,...,...
7963,MR99753,3,C-pre,NaN,labels/MR99753_3_C-pre.nii.gz,mask,MR99753_3_C-pre,images/MR99753_3_C-pre_0000.nii.gz
7964,MR99753,3,DWI,NaN,labels/MR99753_3_DWI.nii.gz,mask,MR99753_3_DWI,images/MR99753_3_DWI_0000.nii.gz
7965,MR99753,3,InPhase,NaN,labels/MR99753_3_InPhase.nii.gz,mask,MR99753_3_InPhase,images/MR99753_3_InPhase_0000.nii.gz
7966,MR99753,3,OutPhase,NaN,labels/MR99753_3_OutPhase.nii.gz,mask,MR99753_3_OutPhase,images/MR99753_3_OutPhase_0000.nii.gz


In [12]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "MR"
    else:
        return "SEG"
    
def map_roi(row):
    if row["file_type"] == "scan":
            return ""
    else:
        return "LESION"
    
index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["ROINames"] = index_df.apply(map_modality, axis=1)

index_df["BodyPartExamined"] = "LIVER"




In [15]:
import numpy as np


with open(temp_data_path / "LLD_MMRI_Annotation.json", 'r') as f:
    json_data = json.load(f)

annotation_dict = json_data["Annotation_info"]

# Convert your existing dataframe to a dict
df_dict = index_df.to_dict('records')

# Phase mapping for InPhase/OutPhase discrepancy
PHASE_MAPPING = {
    'InPhase': 'In Phase',
    'OutPhase': 'Out Phase',
}

def extract_series_key(filepath, sample_id, phase_from_df):
    """Extract series identifier from filepath."""
    # If phase is InPhase or OutPhase, map it
    if phase_from_df in PHASE_MAPPING:
        return f"{sample_id}_{PHASE_MAPPING[phase_from_df]}"
    return f"{sample_id}_{phase_from_df}"

# Build lookup dictionary
lookup = {}
for sample_id, series_list in annotation_dict.items():
    for series in series_list:
        phase = series.get('phase')
        key = f"{sample_id}_{phase}"
        lookup[key] = series
        
        # Aliases for InPhase/OutPhase
        if phase == 'In Phase':
            lookup[f"{sample_id}_InPhase"] = series
        elif phase == 'Out Phase':
            lookup[f"{sample_id}_OutPhase"] = series

# Update each record
for record in df_dict:
    filepath = record.get('filepath', '')
    if not filepath:
        continue
    
    sample_id = record.get('SampleID', '')
    phase_from_df = record.get('Phase', '')
    
    key = extract_series_key(filepath, sample_id, phase_from_df)
    series_data = lookup.get(key)
    
    if series_data:
        # Basic metadata
        pixel_spacing = series_data.get('pixel_spacing', [0, 0])
        slice_spacing = series_data.get('slice_spacing', 0)
        origin = series_data.get('origin', [0, 0, 0])
        
        record['spacing'] = f"({pixel_spacing[0]}, {pixel_spacing[1]}, {slice_spacing})"
        record['origin'] = f"({origin[0]}, {origin[1]}, {origin[2]})"
        
        # Annotation (lesion) data
        annotation = series_data.get('annotation', {})
        lesion_data = annotation.get('lesion', {})
        
        if '0' in lesion_data:
            bbox = lesion_data['0'].get('bbox', {})
            box_3d = bbox.get('3D_box', {})
            
            if box_3d:
                record['mask.bbox.size'] = box_3d.get('area', 0)
                record['mask.bbox.min_coord'] = f"({box_3d.get('x_min', 0)}, {box_3d.get('y_min', 0)}, {box_3d.get('z_min', 0)})"
                record['mask.bbox.max_coord'] = f"({box_3d.get('x_max', 0)}, {box_3d.get('y_max', 0)}, {box_3d.get('z_max', 0)})"
            
            boxes_2d = bbox.get('2D_box', [])
            if boxes_2d:
                record['num_slices_with_lesion'] = len(boxes_2d)
                record['total_2d_area'] = sum([b.get('area', 0) for b in boxes_2d])

# Convert back to dataframe
df_updated = pd.DataFrame(df_dict)

# Fill ImageID
df_updated['ImageID'] = range(1, len(df_updated) + 1)

# Classify the phase (if you want to keep this)
def classify_phase(phase):
    if phase in ['T2WI']:
        return 'T2_Weighted'
    elif phase in ['DWI']:
        return 'Diffusion_Weighted'
    elif phase in ['C-pre']:
        return 'Pre_Contrast'
    elif phase in ['C+A']:
        return 'Arterial_Phase'
    elif phase in ['C+V']:
        return 'Venous_Phase'
    elif phase in ['C+Delay']:
        return 'Delayed_Phase'
    elif phase in ['InPhase', 'OutPhase']:
        return 'T1_Dual_Echo'
    else:
        return 'Unknown'

df_updated['class'] = df_updated['Phase'].apply(classify_phase)

# Reorder columns
desired_columns = [
    'ImageID', 'ROINames', 'filepath', 'file_type', 'Phase', 'SampleID', 'class',
    'hash', 'size', 'ndim', 'nvoxels', 'spacing', 'origin', 'direction',
    'min', 'max', 'sum', 'mean', 'std', 'variance', 'dtype_str',
    'dtype_numpy', 'mask.bbox.size', 'mask.bbox.min_coord',
    'mask.bbox.max_coord', 'mask.feret_diameter', 'mask.roundness',
    'mask.flatness', 'mask.elongation', 'mask.equivalent_spherical_radius',
    'mask.equivalent_spherical_perimeter', 'mask.equivalent_ellipsoid_diameters',
    'mask.volume_count', 'reference_scan', 'Modality', 'BodyPartExamined'
]

existing_cols = [col for col in desired_columns if col in df_updated.columns]
df_final = df_updated[existing_cols]

print(df_final.head())

ImageID ROINames                                filepath file_type  \
0        1       MR      images/MR-391135_1_C+A_0000.nii.gz      scan   
1        2       MR  images/MR-391135_1_C+Delay_0000.nii.gz      scan   
2        3       MR      images/MR-391135_1_C+V_0000.nii.gz      scan   
3        4       MR    images/MR-391135_1_C-pre_0000.nii.gz      scan   
4        5       MR      images/MR-391135_1_DWI_0000.nii.gz      scan   

     Phase   SampleID               class  \
0      C+A  MR-391135      Arterial_Phase   
1  C+Delay  MR-391135       Delayed_Phase   
2      C+V  MR-391135        Venous_Phase   
3    C-pre  MR-391135        Pre_Contrast   
4      DWI  MR-391135  Diffusion_Weighted   

                               spacing                         origin  \
0  (0.7422, 0.7422, 2.599997183098592)  (-199.14, -194.451, -136.739)   
1  (0.7422, 0.7422, 2.599997183098592)  (-199.14, -194.451, -136.739)   
2  (0.7422, 0.7422, 2.599997183098592)  (-199.14, -194.451, -136.739)   
3  (0.7422, 0.7422, 2.599997183098592)  (-199.14, -194.451, -136.739)   
4  (1.4844, 1.4844, 7.999986956521739)  (-202.478, -221.47, -137.075)   

   mask.bbox.size  mask.bbox.min_coord       mask.bbox.max_coord  \
0    33776.849480   (72.8, 217.03, 33)  (127.270004, 279.04, 42)   
1    13564.539808  (74.48, 226.25, 34)  (113.020004, 276.53, 40)   
2    12712.175000  (71.13, 213.68, 34)      (113.86, 273.18, 38)   
3    31851.368366  (73.64, 210.33, 37)   (126.42999, 277.37, 45)   
4     5643.287774   (36.98, 119.2, 10)    (71.34, 160.26001, 13)   

  reference_scan Modality BodyPartExamined  
0            NaN       MR            LIVER  
1            NaN       MR            LIVER  
2            NaN       MR            LIVER  
3            NaN       MR            LIVER  
4            NaN       MR            LIVER

In [16]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet

index_df = df_final
index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

/tmp/ipykernel_458992/920433607.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
/tmp/ipykernel_458992/920433607.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  index_df["SampleID"] = index_df["SampleID"].astype(str)


,ImageID,ROINames,filepath,file_type,Phase,SampleID,class,spacing,origin,mask.bbox.size,mask.bbox.min_coord,mask.bbox.max_coord,reference_scan,Modality,BodyPartExamined
0,1,MR,images/MR-391135_1_C+A_0000.nii.gz,scan,C+A,MR-391135,Arterial_Phase,"(0.7422, 0.7422, 2.599997183098592)","(-199.14, -194.451, -136.739)",33776.849480,"(72.8, 217.03, 33)","(127.270004, 279.04, 42)",NaN,MR,LIVER
1,2,MR,images/MR-391135_1_C+Delay_0000.nii.gz,scan,C+Delay,MR-391135,Delayed_Phase,"(0.7422, 0.7422, 2.599997183098592)","(-199.14, -194.451, -136.739)",13564.539808,"(74.48, 226.25, 34)","(113.020004, 276.53, 40)",NaN,MR,LIVER
2,3,MR,images/MR-391135_1_C+V_0000.nii.gz,scan,C+V,MR-391135,Venous_Phase,"(0.7422, 0.7422, 2.599997183098592)","(-199.14, -194.451, -136.739)",12712.175000,"(71.13, 213.68, 34)","(113.86, 273.18, 38)",NaN,MR,LIVER
3,4,MR,images/MR-391135_1_C-pre_0000.nii.gz,scan,C-pre,MR-391135,Pre_Contrast,"(0.7422, 0.7422, 2.599997183098592)","(-199.14, -194.451, -136.739)",31851.368366,"(73.64, 210.33, 37)","(126.42999, 277.37, 45)",NaN,MR,LIVER
4,5,MR,images/MR-391135_1_DWI_0000.nii.gz,scan,DWI,MR-391135,Diffusion_Weighted,"(1.4844, 1.4844, 7.999986956521739)","(-202.478, -221.47, -137.075)",5643.287774,"(36.98, 119.2, 10)","(71.34, 160.26001, 13)",NaN,MR,LIVER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7963,7964,SEG,labels/MR99753_3_C-pre.nii.gz,mask,C-pre,MR99753,Pre_Contrast,"(0.703125, 0.703125, 3.750000943396226)","(-186.1538, -192.3077, -130.1443)",7433.280000,"(117.35, 176.0, 12)","(152.15, 211.6, 17)",images/MR99753_3_C-pre_0000.nii.gz,SEG,LIVER
7964,7965,SEG,labels/MR99753_3_DWI.nii.gz,mask,DWI,MR99753,Diffusion_Weighted,"(0.95052083333333, 0.95052083333333, 9.3000003...","(-186.7307, -184.7193, -98.57474)",528.639764,"(80.35, 161.4, 3)","(103.95, 183.79999, 3)",images/MR99753_3_DWI_0000.nii.gz,SEG,LIVER
7965,7966,SEG,labels/MR99753_3_InPhase.nii.gz,mask,InPhase,MR99753,T1_Dual_Echo,"(0.703125, 0.703125, 9.300000197785765)","(-184.5981, -181.0858, -121.4788)",928.000000,"(119.15, 162.6, 5)","(148.15, 194.6, 5)",images/MR99753_3_InPhase_0000.nii.gz,SEG,LIVER
7966,7967,SEG,labels/MR99753_3_OutPhase.nii.gz,mask,OutPhase,MR99753,T1_Dual_Echo,"(0.703125, 0.703125, 9.300000197785765)","(-184.5981, -181.0858, -121.4788)",2269.759344,"(117.95, 161.2, 5)","(152.54999, 194.0, 6)",images/MR99753_3_OutPhase_0000.nii.gz,SEG,LIVER
